<a href="https://colab.research.google.com/github/muhammadbinasim33-ui/muhammadbinasim33-ui-.github.io/blob/main/i234600_BSBA_7AAssignment_2_Bookstore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Assignment 2: Book Store Scrapping**

### Step 1: Import libraries and request

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://books.toscrape.com/"

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
print("response",response)


response <Response [200]>


#### Step 2: Scraping atleast 3 pages in a loop

In [9]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

all_books = []

for page in range(1, 4):

    if page == 1:
        url = "https://books.toscrape.com/"
    else:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    for book in soup.select("article.product_pod"):

        title = book.h3.a["title"]

        price_text = book.select_one(".price_color").get_text()
        price = float(
            price_text.replace("£", "").replace("Â", "").strip()
        )

        rating_word = book.select_one(".star-rating")["class"][1]
        rating = rating_map[rating_word]

        availability = book.select_one(
            ".availability"
        ).get_text(" ", strip=True)

        all_books.append({
            "Title": title,
            "Price": price,
            "Rating": rating,
            "Availability": availability
        })

df = pd.DataFrame(all_books)

df.to_csv("competitor_books.csv", index=False)

display(df.head())
print("Total books:", len(df))
print("Dataset saved as competitor_books.csv")

,Title,Price,Rating,Availability
0,A Light in the Attic,51.77,3,In stock
1,Tipping the Velvet,53.74,1,In stock
2,Soumission,50.10,1,In stock
3,Sharp Objects,47.82,4,In stock
4,Sapiens: A Brief History of Humankind,54.23,5,In stock


Total books: 60
Dataset saved as competitor_books.csv


### **Step3: Intelligence Questions**

In [10]:
# 1. Most expensive book
print("Most expensive book:")
display(df.loc[[df["Price"].idxmax()]])

# 2. Cheapest book
print("Cheapest book:")
display(df.loc[[df["Price"].idxmin()]])

# 3. Average book price
print("Average price: £", round(df["Price"].mean(), 2))

# 4. Number of 5-star books
print("5-star books:", (df["Rating"] == 5).sum())

# 5. Percentage of books in stock
in_stock = df["Availability"].str.contains(
    "In stock", case=False
).sum()

percentage = in_stock / len(df) * 100
print("In stock percentage:", round(percentage, 2), "%")

# 6. Top 5 highest-rated books
print("Top 5 highest-rated books:")
display(
    df.sort_values(
        ["Rating", "Price"],
        ascending=[False, True]
    ).head(5)
)

# 7. Cheapest 5-star book
five_star = df[df["Rating"] == 5]

print("Cheapest 5-star book:")
if len(five_star) > 0:
    display(five_star.loc[[five_star["Price"].idxmin()]])
else:
    print("No 5-star books found.")

Most expensive book:


,Title,Price,Rating,Availability
40,Slow States of Collapse: Poems,57.31,3,In stock


Cheapest book:


,Title,Price,Rating,Availability
20,In Her Wake,12.84,1,In stock


Average price: £ 35.0
5-star books: 14
In stock percentage: 100.0 %
Top 5 highest-rated books:


,Title,Price,Rating,Availability
34,Sophie's World,15.94,5,In stock
54,Thirst,17.27,5,In stock
12,Set Me Free,17.46,5,In stock
30,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock
43,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11,5,In stock


Cheapest 5-star book:


,Title,Price,Rating,Availability
34,Sophie's World,15.94,5,In stock


# **Intelligence Question 1: Which Scraped book is the most expensive?**
### Answer: The most expensive book is: "Slow States of Collapse Poems" with a price of 57.31.
# **Intelligence  Question 2: Which scraped book is cheapest?**
### Answer: The cheapest book is:" In Her Wake" with a rating of one and is available.

# **Intelligence Question 3: What is the average book price?**
### Answer: the average book price in the whole store is 35 pounds.

## **Intelligence Question 4: How many books have a five star rating?**
## Answer: There are only 14 books which are rated 5 star.
### **Intelligence Question 5: What percentage of the scarped books are in stock?**
### Answer: 100 percent are in stock.
#### Intelligence Question 6: What are the top 5 highest rated books?
Answer: the top five rated are:


1.  Sophie's World
2.  Thirst
3. Set Me Free
4.  The Four Agreements
5. The Higher Selfie
## **Intelligence Question 7: Among the five star books which one is the cheapest?**
## Answer: Sophie's World.



Step 4: Business Question: If I was launching a bookstore online which books will I prioritize and why from the data set? Add two evidences from the dataset.
Answer: I will prioritize the top five star books with cheaper prices. Example Sophie's World and Thirst

# Step 5: CEO Challenge:

In [11]:
budget = 10000  # £100 in pence

prices = (df["Price"] * 100).round().astype(int).tolist()
ratings = df["Rating"].astype(int).tolist()

n = len(df)

# Best rating score possible for each spending amount
best = [0] * (budget + 1)

# Record which books were selected
take = [[False] * (budget + 1) for _ in range(n)]

for i in range(n):
    price = prices[i]
    rating = ratings[i]

    for money in range(budget, price - 1, -1):
        new_score = best[money - price] + rating

        if new_score > best[money]:
            best[money] = new_score
            take[i][money] = True

# Find the spending amount with the highest score
money = max(range(budget + 1), key=lambda x: best[x])

selected = []

# Reconstruct the selected books
for i in range(n - 1, -1, -1):
    if take[i][money]:
        selected.append(i)
        money -= prices[i]

selected.reverse()

chosen = df.iloc[selected].copy()

print("Selected books:")
display(chosen)

print("Total cost: £", round(chosen["Price"].sum(), 2))
print("Total rating score:", chosen["Rating"].sum())
print("Budget remaining: £", round(100 - chosen["Price"].sum(), 2))

Selected books:


,Title,Price,Rating,Availability
10,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99,2,In stock
12,Set Me Free,17.46,5,In stock
30,The Four Agreements: A Practical Guide to Pers...,17.66,5,In stock
34,Sophie's World,15.94,5,In stock
47,Untitled Collection: Sabbath Poems 2014,14.27,4,In stock
54,Thirst,17.27,5,In stock


Total cost: £ 96.59
Total rating score: 26
Budget remaining: £ 3.41


the total cost of the selected books from a 100 is 96.59 with a rating score of 26 and a budget remaining from 1000 is 3.41 pounds.

**Assignment Progress:**


---


*   Scraper working for the first page : Yes
*   Scraper working for atleast 3 pages: Yes
*   Data frame contains Title, Price, Rating and Availability: Yes
*   competitor_books.csv saved: Yes
*   All 7 questions Answered: Yes
*   Business Recommendation: Yes
*   Python file saved: Yes
*   Scraper uploaded to Github and copied link: Yes
*   Linkedin post with github link attached: Yes











